In [3]:
import duckdb, pandas as pd, lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

con = duckdb.connect("../data/fraud.duckdb", read_only=True)
df = con.execute("SELECT * FROM train ORDER BY TransactionDT").df()

# save memory
for c in df.select_dtypes("float64").columns:
    df[c] = df[c].astype("float32")

drop = ["isFraud", "TransactionID", "TransactionDT"]
features = [c for c in df.columns if c not in drop]
for c in features:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("category")

# time-based split: earliest 80% train, latest 20% validation
cut = df["TransactionDT"].quantile(0.8)
train, valid = df[df["TransactionDT"] <= cut], df[df["TransactionDT"] > cut]

model = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=64,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1,
)
model.fit(train[features], train["isFraud"])

p = model.predict_proba(valid[features])[:, 1]
print("Fraud rate in validation:", round(valid["isFraud"].mean(), 4))
print("PR-AUC :", round(average_precision_score(valid["isFraud"], p), 4))
print("ROC-AUC:", round(roc_auc_score(valid["isFraud"], p), 4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fraud rate in validation: 0.0344
PR-AUC : 0.5553
ROC-AUC: 0.9125
